In [9]:
import pandas as pd
import table_convert
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage
from tqdm.notebook import tqdm as log_progress
import os
import re
import time
import cga_utils
from langchain_community.chat_models import ChatOllama

In [ ]:
messages = [("human","""
Given the following set of annotated values:
{value_list}
and the following question:
{question} 
Analyse the following Python code, which is generated by an LLM. 
It makes a selection error mistake while computing the answer to the question. 
What is the problem with he code?
CODE:
{code}
/no_think"""
)]

In [23]:
llm = ChatOllama(model="qwen3:4b", temperature = 0.0, top_p = 1, repeat_penalty=1, presence_penalty=0, frequency_penalty=0)  

messages = [("human","""
Given the following set of annotated values:
{value_list}
and the following question:
{question} 
Analyse the following Python code, which is generated by an LLM. 
It makes a selection error mistake while computing the answer to the question. 
What is the problem with he code?
Do NOT generate code, do NOT calculate the answer, just answer with one sentence what is the problem. 
CODE:
{code}
/no_think"""
)]




dataset = pd.read_json('dataset_raw/tatqa_dataset_dev.json')    

_table, q_block =  cga_utils.get_question(dataset, "6c44a1a8-0785-43a0-90ab-7e21df2c57d9")
table = _table['table']
value_list = table_convert.convert_multitable(table)    
question = q_block["question"]

code = """
def run(value_list):
    # Filter for 'Expected return on plan assets' and 'International'
    international_expected_return = [v['number_value'] for v in value_list if v['category'] == 'Expected return on plan assets' and v['header3'] == 'International']
    
    # Extract 2018 and 2019 values
    value_2018 = next(v['number_value'] for v in value_list if v['header1'] == '2018' and v['header3'] == 'International')
    value_2019 = next(v['number_value'] for v in value_list if v['header1'] == '2019' and v['header3'] == 'International')
    
    # Calculate percentage change
    percentage_change = ((value_2019 - value_2018) / value_2018) * 100
    
    # Return as float with two decimal places and 'percent' scale
    return (round(percentage_change, 2), 'percent')
"""



In [67]:
llm = ChatOllama(model="qwen3:4b", temperature = 0.0, top_p = 1, repeat_penalty=1, presence_penalty=0, frequency_penalty=0)  

from langchain.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path=".langchain.ollama38.db"))

messages = [("human","""
Given the following set of annotated values:
{value_list}
and the following question:
{question} 
Analyse the following Python code. 
It makes an {error_type} mistake while computing the answer to the question. 
The expected computation is '{derivation}'.
What is the problem with he code?
Do NOT generate code, do NOT calculate the answer, answer with one sentence what the problem is. 
CODE:
{code}
/no_think"""
)]

In [82]:
def get_problem(messages, value_list, question, code, error_type, derivation, scale):
    prompt = ChatPromptTemplate.from_messages(messages)
    
    output_parser = StrOutputParser()
    
    chain = prompt | llm | output_parser
    
    response = chain.invoke({"value_list": value_list, "question":question, "code": code, "error_type": error_type, "derivation": derivation, "scale":scale})
    return response

In [83]:

report = pd.read_csv("res/e38_25.csv")
res = []
for i, item in log_progress(report.iterrows()):   
    problem = get_problem(messages, item["value_list"], item["question"], item["code"], item["error_code"], item["derivation"], item["scale"])  
    if not item["exact_match"]:
        res.append({"qid": item["qid"], "problem": problem.replace("<think>\n\n</think>\n\n", "")})

0it [00:00, ?it/s]

KeyboardInterrupt: 

In [ ]:
messages = [("human","""
Given the following questions:
{value_list}
and the following question:
{question} 
Analyse the following Python code. 
It makes a mistake while computing the answer to the question. 
The expected computation is '{derivation}'.
The expected scale is '{scale}'.
The analyzed code is expected to return the tuple (numeric_value, scale)
What is the problem with he code?
Do NOT generate code, do NOT calculate the answer, answer with one sentence what the problem is. 
The answer must be short and general'. CAN NOT include details of the code, etc. years, column names.  
After the answer, generate a problem type TAG. 
Output format:  (problem, problem_type)
CODE:
{code}
/no_think"""
)]


In [107]:
messages = [("human","""
Given the following set of annotated values:
{value_list}
and the following question:
{question} 
Analyse the following Python code. 
It makes a mistake while computing the answer to the question. 
The expected computation is '{derivation}'.
The expected scale is '{scale}'.
The analyzed code is expected to return the tuple (numeric_value, scale)
What is the problem with he code?
Do NOT generate code, do NOT calculate the answer, answer with one sentence what the problem is. 
The answer must be short and general'. CAN NOT include details of the code, etc. years, column names.  
After the answer, generate a problem type TAG. 
Output format:  (problem, problem_type)
CODE:
{code}
/no_think"""
)]

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
res = []
for i, item in log_progress(errors.iterrows()):   
    problem = get_problem(messages, item["value_list"], item["question"], item["code"], item["error_code"], item["derivation"], item["scale"])  
    if not item["exact_match"]:
        problem = problem.replace("<think>\n\n</think>\n\n", "")
        print(problem)
        res.append({"qid": item["qid"], "problem": problem})
pd.DataFrame(res).to_csv('res/error_summary_e38_v18r.csv')      

0it [00:00, ?it/s]

(problem, problem_type) = ("The code is summing the 2019 values for 'Defined contribution schemes' but is not taking the average, it is returning the total instead of the average.", "incorrect_aggregation")
(problem, problem_type) = ("The code is not correctly calculating the average as it sums the values and divides by the count, but the expected computation is (57+44)/2", "incorrect_calculation")
(problem, problem_type) = ("The code incorrectly computes the difference between the 2019 values of defined contribution schemes and defined benefit schemes without taking their average", "incorrect_average_computation")
(problem, problem_type) = ("The code only returns the 2019 free cash flow value without averaging it with the 2018 value", "incorrect_calculation")
(problem, problem_type) = ("The code returns only the 2018 free cash flow value instead of the average of 2017 and 2018", "incorrect_calculation")
(problem, problem_type) = ( "The code does not compute the average free cash flow 

In [102]:
res

[{'qid': 'a0414f81-8dc2-44b2-a441-2c9d9c805c4d',
  'problem': '(problem, problem_type) = ("The code is summing the 2019 values for \'Defined contribution schemes\' but is not taking the average, it is returning the total instead of the average.", "incorrect_aggregation")'},
 {'qid': 'bf7abd62-d9cd-48d2-8826-1457684019a3',
  'problem': '(problem, problem_type) = ("The code is not correctly calculating the average as it sums the values and divides by the count, but the expected computation is (57+44)/2", "incorrect_calculation")'},
 {'qid': '4d259081-6da6-44bd-8830-e4de0031744c',
  'problem': '(problem, problem_type) = ("The code incorrectly computes the difference between the 2019 values of defined contribution schemes and defined benefit schemes without taking their average", "incorrect_average_computation")'},
 {'qid': 'dc5e217a-a7b3-4fc9-ac0f-13d328f26b20',
  'problem': '(problem, problem_type) = ("The code only returns the 2019 free cash flow value without averaging it with the 2018

# summary

In [120]:
messages = [("human","""
Summarize the following statement about a program code, that makes a calculation error. 
The summarization has to focus on the solution to fix the fail, and cannot be longer than 5 words. 
Example: use both years for assition.
statement:
{statement}
/no_think"""
)]

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
res2 = []
for row in log_progress(res):   
    item = row['problem'].replace('(problem, problem_type) = ', '')
    prompt = ChatPromptTemplate.from_messages(messages)
    
    output_parser = StrOutputParser()
    
    chain = prompt | llm | output_parser
    
    summary = chain.invoke({"statement": item})
    
    summary = summary.replace("<think>\n\n</think>\n\n", "").replace("fix", "").replace("Fix", "").strip()
    print(summary)
    res2.append({"qid": row["qid"], "problem": problem, "summary":summary })
pd.DataFrame(res2).to_csv('res/error_summary2_e38_v18r.csv') 

  0%|          | 0/187 [00:00<?, ?it/s]

summing instead of averaging
Divide sum by count.
average computation.
average both years for calculation.
calculate average of 2017 and 2018.
use both years for average.
average difference computation.
expenses calculation.
Summarize the error in data extraction.
line identification error.
scale mismatch error.
Set scale to percent.
scale_mismatch
scale mismatch.
use actual cash change.
Use correct headers for tax expense.
Use correct tax expense term.
average tax calculation.
scale inconsistency.
Apply correct scale to average.
scale mismatch.
year filtering error.
Set scale to percent.
Multiply by correct category.
Use 2017 shares in numerator.
Use correct price extraction.
incorrect_scale error.
Apply correct scale to result.
scale mismatch error.
Apply correct scale to result.
Check 'CHANGE' in correct field.
Missing 2021-2022 in average.
Use 'percent' instead of 'thousand'.
Use 'percent' for scale.
Account for category in sum.
scale conversion missing.
Use 'scale_mismatch' for er

In [113]:
messages = [("human","""
Summarize the following statement about a program code, that makes a calculation error. 
The summarization has to focus on the solution to fix the fail, and cannot be longer than 6 words. 
statement:
{statement}
/no_think"""
)]

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
res2 = []
for row in log_progress(res):   
    item = row['problem'].replace('(problem, problem_type) = ', '')
    prompt = ChatPromptTemplate.from_messages(messages)
    
    output_parser = StrOutputParser()
    
    chain = prompt | llm | output_parser
    
    summary = chain.invoke({"statement": item})
    
    summary = summary.replace("<think>\n\n</think>\n\n", "")
    print(summary)
    res2.append({"qid": row["qid"], "problem": problem, "summary":summary })
pd.DataFrame(res2).to_csv('res/error_summary2_e38_v18r.csv')

  0%|          | 0/187 [00:00<?, ?it/s]

Summarize: Fix average calculation for 2019 values.
Divide sum by count to fix error.
Fix average computation error.
Fix calculation by averaging 2018 and 2019 values.
Fix calculation to include both years.
Use correct average calculation for each year.
Fix average difference calculation.
Fix computation to include expenses.
Fix data extraction to sum correct assets.
Fix identification of 'As reported' line.
Fix scale mismatch error.
Scale not set to percent; set scale to percent.
Scale mismatch; fix value scaling.
Fix scale mismatch error.
Fix percentage calculation error.
Use correct headers for current tax expense.
Use correct tax expense term for calculation.
Fix mathematical computation error.
Fix scale inconsistency for 2018 value.
Scale mismatch in average calculation.
Fix scale mismatch in return statement.
Fix data filtering for 2017 and 2018.
Set scale to "percent" to fix error.
Fix data handling error in multiplication.
Use 2017 shares in numerator.
Fix value extraction meth

In [126]:
messages = [("human","""
Given the following program code. 
Extract the caalculation pattern the code implements. The calculation part is at the end of the code. 
In pattern, hashmark represents a variable. 
Ex. if it adds two numbers, the pattern is #+#
Ex. if it adds three numbers, the pattern is #+#+#
if it subtracts two numbers, the pattern is #-#
if it averages two number, the pattern is (#+#)/#
DO NOT write explanation, just the pattern.
Output format: pattern
CODE:
{code}
/no_think"""
)]

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
res3 = []
for i, item in log_progress(errors.iterrows()):      
    
    prompt = ChatPromptTemplate.from_messages(messages)
    
    output_parser = StrOutputParser()
    
    chain = prompt | llm | output_parser
    
    pattern = chain.invoke({"code": item["code"]})
    
    pattern = pattern.replace("<think>\n\n</think>\n\n", "")
    print(pattern)
    res2.append({"qid": row["qid"], "pattern": pattern })
pd.DataFrame(res2).to_csv('res/error_summary3_e38_v18r.csv')

0it [00:00, ?it/s]

pattern
(#+#)/#
pattern
(#+#)/#
#-#
pattern
(#+#)/#
pattern: #+#
#-#
pattern
(#+#)/#
pattern: #+#
pattern
(#+#)/#
#+#
(-#-#)
#-#/#*#
pattern
(#+#)/#
pattern
( #-# )
pattern: #+#
(#+#)/#
(#+#)/#
pattern
#+#
pattern
(#+#)/#
(#+#)/#
(#+#)/#
#-#/#
(#+#)*#
#/+#+#
pattern
(#+#)/#
(#+#)/#
#-#/#
pattern
#-#
#-#/#*100
pattern: #+#+#
pattern
# + # + # + # / #
%-#
%-#
#/#
#/#
#-#/#*#
(#+#)/#
#/#
#-#
(#+#)/#
pattern
(#+#)-#
%-#
(#+#)/#
pattern
(#+#)/#
#-#
#-#/#*100
#-#
#-#/#
#-#
#-#
pattern
(#+#)/#
pattern
(#+#)/#
pattern
(#+#)/#
#/#*#
pattern
#-#
(#+#)/#
(#+#)/#
#-#
(#+#)/#
#-#/#
pattern
#-#
pattern
(#+#)-#
(#+#)/#
#-#
#-#
(#+#)/#
(#+#)/#


KeyboardInterrupt: 

['((#+#)/#)']


In [72]:
messages_2 = [("human","""
The following list contains statements about bugs in various Python code. Codes are making computations on data located in a list of annotated values. The code usually selects the relevant dataa, and implements the needed computation. 
Identify the top 5 types of bugs in the list. 
{bugs}
/no_think"""
)]

prompt_2 = ChatPromptTemplate.from_messages(messages_2)
    
output_parser = StrOutputParser()

chain = prompt_2 | llm | output_parser

response = chain.invoke({"bugs":  pd.DataFrame(res)["problem"]})
print(response)

<think>

</think>

Based on the list of bug descriptions provided, here are the **top 5 types of bugs** identified:

1. **Incorrect Data Filtering or Selection**  
   - Examples: "The code only includes 2019 and 2018 but does not include 2020", "The code incorrectly includes the '3-5 Years' category", "The code incorrectly uses the total balances for a different purpose".

2. **Incorrect Use of Built-in Functions**  
   - Example: "The code incorrectly uses `next()` to retrieve data", "The code incorrectly uses the 'Vested' category".

3. **Incorrect Calculation or Logic**  
   - Examples: "The code incorrectly calculates the average of...", "The code incorrectly uses the 2019 non-vested balances".

4. **Scale or Range Issues**  
   - Example: "The code does not account for the scale of the data", "The code incorrectly uses the total balances for a different purpose".

5. **Incorrect Categorization or Data Mapping**  
   - Examples: "The code incorrectly uses the 'Vested' category", "T

In [61]:
messages_2 = [("human","""
The following list contains statements about bugs in various Python code. Codes are making computations on data located in a list of annotated values. The code usually selects the relevant dataa, and implements the needed computation. 
Identify the top 5 types of bugs in the list. 
{bugs}
/no_think"""
)]

prompt_2 = ChatPromptTemplate.from_messages(messages_2)
    
output_parser = StrOutputParser()

chain = prompt_2 | llm | output_parser

response = chain.invoke({"bugs":  pd.DataFrame(res)["problem"]})
print(response)

<think>

</think>

Based on the list of bug descriptions provided, here are the **top 5 types of bugs** identified:

1. **Incorrect use of `next()`**  
   - Multiple entries (e.g., 0, 143) indicate that the code is using `next()` improperly, possibly to extract data from an iterator or list without proper context or handling.

2. **Incorrect handling of the 'scale' field**  
   - Entry 1 and 144 suggest that the code is not properly accounting for or using the 'scale' field, which may be critical for calculations or data interpretation.

3. **Incorrect category usage**  
   - Entries 2 and 146 point to the code using the same category ('F') or incorrect categories like "Vested" or "thousand" in a way that leads to incorrect results.

4. **Incorrect calculation of "change" or difference**  
   - Entries 3 and 4 indicate that the code is miscalculating the "change" or difference between values, likely due to incorrect logic or data selection.

5. **Incorrect inclusion of specific data (e

In [73]:
messages_2 = [("human","""
The following list contains statements about bugs in various Python code. Codes are making computations on data located in a list of annotated values. The code usually selects the relevant dataa, and implements the needed computation. 
Identify the top 5 types of bugs in the list, and according to them give 5 code generation instruction to the LLM generating the code that enhances code logic. 
{bugs}
/no_think"""
)]

prompt_2 = ChatPromptTemplate.from_messages(messages_2)
    
output_parser = StrOutputParser()

chain = prompt_2 | llm | output_parser

response = chain.invoke({"bugs":  pd.DataFrame(res)["problem"]})
print(response)

<think>

</think>

Based on the provided list of bug descriptions, here's an analysis of the **top 5 types of bugs** and **5 code generation instructions** to enhance code logic when generating Python code for data processing tasks:

---

### **Top 5 Types of Bugs:**

1. **Incorrect Data Filtering or Selection**  
   - Example: The code only includes 2019 and 2018 but does not account for other years, or it includes incorrect categories like '3-5 Years' or 'Vested'.

2. **Incorrect Use of Built-in Functions**  
   - Example: The code incorrectly uses `next()` to retrieve data, which may not be the correct approach for iteration or data retrieval.

3. **Incorrect Calculation of Averages or Metrics**  
   - Example: The code incorrectly calculates the average of values, possibly due to incorrect data selection or aggregation.

4. **Misuse of Data Categories or Labels**  
   - Example: The code incorrectly includes or excludes certain categories (e.g., 'Vested', 'Non-Vested') or misinterp

In [62]:
messages_2 = [("human","""
The following list contains statements about bugs in various Python code. Codes are making computations on data located in a list of annotated values. The code usually selects the relevant dataa, and implements the needed computation. 
Identify the top 5 types of bugs in the list, and according to them give 5 code generation instruction to the LLM generating the code that enhances code logic. 
{bugs}
/no_think"""
)]

prompt_2 = ChatPromptTemplate.from_messages(messages_2)
    
output_parser = StrOutputParser()

chain = prompt_2 | llm | output_parser

response = chain.invoke({"bugs":  pd.DataFrame(res)["problem"]})
print(response)

<think>

</think>

Based on the list of bug descriptions, we can identify the **top 5 types of bugs** in the Python code, and then provide **5 code generation instructions** to enhance the logic of the generated code.

---

### **Top 5 Types of Bugs Identified:**

1. **Incorrect use of `next()`**  
   - The code uses `next()` in a way that may not correctly retrieve the intended value from an iterator or list.

2. **Incorrect handling of the 'scale' field**  
   - The code fails to properly account for or use the 'scale' field in the data, leading to incorrect computations.

3. **Incorrect category usage**  
   - The code uses the same category label (e.g., 'F') for different types of data, leading to incorrect grouping or processing.

4. **Incorrect calculation of "change" or difference**  
   - The code computes the "change" or difference between values incorrectly, possibly due to incorrect indexing or data selection.

5. **Incorrect inclusion of specific data entries**  
   - The c

In [74]:
report = pd.read_csv("res/e38_18.csv")
res = []
for i, item in log_progress(report.iterrows()):   
    problem = get_problem(messages, item["value_list"], item["question"], item["code"], item["error_code"], item["derivation"])  
    if not item["exact_match"]:
        res.append({"qid": item["qid"], "problem": problem.replace("<think>\n\n</think>\n\n", "")})

0it [00:00, ?it/s]

In [75]:
messages_2 = [("human","""
The following list contains statements about bugs in various Python code. Codes are making computations on data located in a list of annotated values. The code usually selects the relevant dataa, and implements the needed computation. 
Identify the top 5 types of bugs in the list, and according to them give 5 code generation instruction to the LLM generating the code that enhances code logic. 
{bugs}
/no_think"""
)]

prompt_2 = ChatPromptTemplate.from_messages(messages_2)
    
output_parser = StrOutputParser()

chain = prompt_2 | llm | output_parser

response = chain.invoke({"bugs":  pd.DataFrame(res)["problem"]})
print(response)

<think>

</think>

Based on the list of bug descriptions provided, here's an analysis of the **top 5 types of bugs** and **5 code generation instructions** to enhance code logic when generating Python code for this kind of problem:

---

### **Top 5 Types of Bugs:**

1. **Incorrect Inclusion of Entries**  
   - The code includes all entries with certain conditions, but it should only include specific ones (e.g., based on time, value, or category).

2. **Incorrect Summation and Counting**  
   - The code sums or counts all items, but it should only consider relevant subsets of the data (e.g., only certain years, certain categories, or certain values).

3. **Incorrect Calculation of Differences**  
   - The code computes differences between values, but it may be using the wrong values or not accounting for time or category differences.

4. **Incorrect Handling of Scale or Units**  
   - The code may not properly handle units or scales (e.g., percentages, years, or currency), leading to i

In [76]:
 pd.DataFrame(res)

,qid,problem
0,a0414f81-8dc2-44b2-a441-2c9d9c805c4d,The code incorrectly includes all entries with...
1,bf7abd62-d9cd-48d2-8826-1457684019a3,The code incorrectly sums and counts all items...
2,4d259081-6da6-44bd-8830-e4de0031744c,The code incorrectly calculates the difference...
3,263d03ec-83d2-48df-8376-1a72167798f7,The code correctly identifies the values for P...
4,dc5e217a-a7b3-4fc9-ac0f-13d328f26b20,The code incorrectly averages the three years ...
...,...,...
193,890e40b2-3009-449f-b806-49d0a4fa82cd,The code does not correctly handle the scale o...
194,6ce427fa-d1ff-481d-b6ba-7950d43e7c22,The code incorrectly includes 'Less than 1 Yea...
195,5b48fea5-61de-401b-aea9-b2b90b7a0eeb,The code incorrectly uses the 2018 non-vested ...
196,8a7ef462-5a25-48c9-8a49-3fb543a73785,The code incorrectly uses the 2019 non-vested ...
